# PyTorch module script creation

The purpose of this notebook is to create a package of utility scripts for PyTorch modeling purposes.

* `README.md` - Readme file to dictate the purpose & scope of the generated model
* `data.py` - File for preparation & preprocessing of data
    * Download data - remote URLs
    * Preprocessing - arrange data for preprocessing
    * Create various types of datasets & dataloaders for ingestion
        * Data in pre-sorted directories
        * Individual data entries (e.g. single images)
* `model_builder.py` - File for creating PyTorch Model(s)
    * `model_utils.py` - File used for building various utilities involved in modeling, such as feature extractors
* `engine.py` - File for training functions
    * Tracks Loss & Accuracy by default, implement other metrics as needed
* `train.py` - File for executing model training process
* `stats.py` - File for storing model statistics & infomation/display functions after training
* `utils.py` - File for miscellaneous utilities
* `app.py` - File for launching Gradio demo/deployment

Singular execution of the model occurs in `train.py`, which can be run from the commandline within the first directory level using `python train.py`

Gradio demo/deployment launches via `app.py`

Main inspiration from [05. Going Modular Part 2 (script mode)](https://github.com/mrdbourke/pytorch-deep-learning/blob/main/going_modular/05_pytorch_going_modular_script_mode.ipynb).

* **Extra:** `predictions.py` - a file for making predictions with a trained PyTorch model and input image (the main function, `pred_and_plot_image()` was originally created in [06. PyTorch Transfer Learning section 6](https://www.learnpytorch.io/06_pytorch_transfer_learning/#6-make-predictions-on-images-from-the-test-set)).

In [ ]:
import os
from huggingface_hub import InferenceClient

from dotenv import load_dotenv
load_dotenv()
HF_TOKEN = os.environ["HF_TOKEN"]

client = InferenceClient(
    provider="hf-inference",
    api_key=os.environ["HF_TOKEN"],
)

In [ ]:
result = client.translation(
    "Меня зовут Вольфганг и я живу в Берлине",
    model="facebook/mbart-large-50-many-to-many-mmt",
    tgt_lang="en_XX",
    src_lang="ru_RU",
)

print(result)

client = InferenceClient(
    provider="hf-inference",
    api_key=os.environ["HF_TOKEN"],
)

result = client.translation(
    "Меня зовут Вольфганг и я живу в Берлине",
    model="google-t5/t5-base",
)

print(result)

TranslationOutput(translation_text='My name is Wolfgang and I live in Berlin')


In [ ]:
tr_text = "student simon 移动的三百块"
result = client.translation(
    tr_text,
    model="google-t5/t5-large",
    # src_lang="cn_XX",
    # tgt_lang="en_XX",
)
print(result)

tr_text2 = "赢家的视角"
result2 = client.translation(
    tr_text2,
    model="google-t5/t5-large",
    # src_lang="cn_XX",
    # tgt_lang="en_XX",
)
print(result2)

In [5]:
from huggingface_hub import hf_hub_download
from datasets import load_dataset, Audio

dataset = load_dataset("PolyAI/minds14", "en-US", split="train")

In [ ]:
# Initialize containter module directory
from pathlib import Path

# Change below directory name as necessary
model_dir = "TranslationModel" + "/"
Path(model_dir).mkdir(parents=True, exist_ok=True)

In [ ]:
# Script Filenames (Change as needed)
readme = model_dir + "README.md"
data_script = model_dir + "data_fn.py"
model_builder_script = model_dir + "model_builder.py"
model_utils_scripts = model_dir + "model_utils.py"
engine_script = model_dir + "engine.py"
train_script = model_dir + "train.py"
stats_script = model_dir + "stats.py"
utils_script = model_dir + "utils.py"
app_script = model_dir + "app.py"
requirements_script = model_dir + "install_requirements.py"
requirements_file = model_dir + "requirements.txt"  # For deployment imports

In [ ]:
%%writefile {readme}

# README 
This package is intended to serve as an implementation of an Image-to-Translated-Text service, to be coordinated and uploaded via Discord Bot.

In [ ]:
%%writefile {data_script}
"""
Scriptfile for all data retrieval, processing, and loading functions
"""
# Imports
import os
import random
import requests
import tarfile
import zipfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
from tqdm.auto import tqdm

import torch
import torchvision
import torchvision.transforms.functional as F
from torch.utils.data import DataLoader, Dataset, Subset, WeightedRandomSampler
from torchvision import datasets, transforms
from torchvision.datasets import ImageFolder

import numpy as np
from sklearn.model_selection import train_test_split
from PIL import Image
import matplotlib.pyplot as plt

# Global settings
import platform

if platform.system() == "Windows":
    NUM_WORKERS = 0
else:
    # NUM_WORKERS = os.cpu_count()
    NUM_WORKERS = 0


import requests
import os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm


# Download latest lol champion loading splashes
def download_lol_dataset(data_dir: str = "./data/lol_champions"):
    data_directory = Path(data_dir)
    os.makedirs(data_directory, exist_ok=True)

    missing_file = data_directory / "missing_skins.txt"

    # Load previously missing skins (skip them)
    previously_missing = set()
    if missing_file.exists():
        with open(missing_file, "r") as f:
            previously_missing = {line.strip() for line in f.readlines()}

    # Step 1: get latest version
    versions = requests.get(
        "https://ddragon.leagueoflegends.com/api/versions.json"
    ).json()
    latest = versions[0]
    print(
        f"[INFO] Downloading League of Legends champion portraits to {data_dir} using latest version: {latest}."
    )

    # Step 2: get champion list
    champ_list = requests.get(
        f"https://ddragon.leagueoflegends.com/cdn/{latest}/data/en_US/champion.json"
    ).json()
    champions = champ_list["data"].keys()

    # Build list of all download jobs
    jobs = []
    new_missing = []

    for champ in champions:
        champ_dir = data_directory / champ
        os.makedirs(champ_dir, exist_ok=True)

        detail = requests.get(
            f"https://ddragon.leagueoflegends.com/cdn/{latest}/data/en_US/champion/{champ}.json"
        ).json()

        skins = detail["data"][champ]["skins"]

        for skin in skins:
            skin_id = skin["num"]
            file_name = f"{champ}_{skin_id}.jpg"
            url = f"https://ddragon.leagueoflegends.com/cdn/img/champion/loading/{file_name}"
            dest = champ_dir / file_name

            # Skip if file exists or previously known missing
            if dest.exists() or file_name in previously_missing:
                continue

            jobs.append((url, dest, file_name))

    if not jobs:
        print("[INFO] All files already downloaded.")
        return data_directory

    # Parallel download with ETA progress bar
    with ThreadPoolExecutor(max_workers=32) as executor:
        futures = {
            executor.submit(requests.get, url): (url, dest, file_name)
            for url, dest, file_name in jobs
        }

        with tqdm(
            total=len(futures),
            desc="Downloading champion splashes...",
            dynamic_ncols=True,
        ) as pbar:
            for future in as_completed(futures):
                url, dest, file_name = futures[future]
                response = future.result()

                if response.status_code == 200:
                    with open(dest, "wb") as f:
                        f.write(response.content)
                else:
                    new_missing.append(file_name)

                pbar.update(1)

    # Append new missing skins to file
    if new_missing:
        with open(missing_file, "a") as f:
            for m in new_missing:
                f.write(m + "\n")

    print(f"\n[INFO] Download complete, champion portraits saved to {data_dir}.")
    return data_directory


# Download datasets for testing
def download_datasets(data_dir: str = "./data"):
    """Downloads sample testing datasets"""
    # Example code for downloading datasets (uncomment and modify as necessary)

    tv_datasets = [
        # Caltech
        {"ds": datasets.Caltech101, "args": {}},
        {"ds": datasets.Caltech256, "args": {}},
        # EMNIST
        {"ds": datasets.EMNIST, "args": {"split": "balanced", "train": True}},
        # FashionMNIST
        {"ds": datasets.FashionMNIST, "args": {"train": True}},
        {"ds": datasets.FashionMNIST, "args": {"train": False}},
        # Flowers102
        {"ds": datasets.Flowers102, "args": {"split": "train"}},
        # Food101
        {"ds": datasets.Food101, "args": {"split": "train"}},
        {"ds": datasets.Food101, "args": {"split": "test"}},
        # MNIST
        {"ds": datasets.MNIST, "args": {"train": True}},
        {"ds": datasets.MNIST, "args": {"train": False}},
        # QMNIST
        {"ds": datasets.QMNIST, "args": {"what": "train"}},
        # SEMEION
        {"ds": datasets.SEMEION, "args": {}},
        # SVHN
        {"ds": datasets.SVHN, "args": {"split": "train"}},
    ]

    for cfg in tv_datasets:
        ds = cfg["ds"]
        args = cfg["args"]

        print(f"[INFO] Downloading {ds.__name__} with args={args}...")
        ds(root=data_dir, download=True, transform=None, **args)

    print(f"[INFO] All datasets downloaded.")

    return


# Function to check dataloader
def check_dataloader(data_dir: str, dataloader: torch.utils.data.DataLoader):
    print("Verifying dataloader against dataset...")
    random.seed()

    ds = dataloader.dataset
    if isinstance(ds, torch.utils.data.Subset):
        base_ds = ds.dataset
        indices = ds.indices
    else:
        base_ds = ds
        indices = range(len(ds))

    path_obj = data_dir if hasattr(data_dir, "suffix") else Path(data_dir)
    file_count = None

    if path_obj.is_file():
        if path_obj.suffix == ".json":
            with open(path_obj, "r") as f:
                data = json.load(f)
            file_count = sum(len(v) for v in data.values())
            print(f"JSON total images: {file_count}")

        elif path_obj.suffix == ".txt":
            with open(path_obj, "r") as f:
                lines = [line.strip() for line in f if line.strip()]
            file_count = len(lines)
            print(f"TXT total images: {file_count}")

        else:
            print(f"Unsupported file type: {path_obj.suffix}")
    else:
        print(f"Using: {data_dir}")

    print(f"Dataset Length: {len(ds)}")
    print(f"Dataloader length: {len(dataloader)}")

    n = 4
    idxs = random.sample(range(len(ds)), n)

    plt.figure(figsize=(4 * n, 4))

    for i, dataset_i in enumerate(idxs):
        img, label = ds[dataset_i]

        plt.subplot(1, n, i + 1)

        # convert tensor to PIL if needed
        if isinstance(img, torch.Tensor):
            img = F.to_pil_image(img)

        parent = None
        filename = None

        # build title
        if hasattr(ds, "file_list"):
            rel_path, _ = ds.samples[dataset_i]
            parent = rel_path.split("/")[0]
            filename = rel_path.split("/")[-1]
            title = f"{parent}/{filename}"

        elif isinstance(ds, torchvision.datasets.ImageFolder):
            full_path, _ = ds.samples[dataset_i]
            parent = os.path.basename(os.path.dirname(full_path))
            filename = os.path.basename(full_path)
            title = f"{parent}/{filename}"

        else:
            title = f"Label {label}"

        if parent is not None:
            print(f"Index: {dataset_i}\tImage Class: {parent}")
        else:
            print(f"Index: {dataset_i}\tLabel: {label}")

    #     plt.imshow(img)
    #     plt.title(title)
    #     plt.axis("off")

    # plt.tight_layout()
    # plt.show(block=False)
    # plt.pause(0.001)

    return


class ListToDataset(Dataset):
    def __init__(self, root, file_list, transform=None):
        self.root = root
        self.file_list = file_list
        self.transform = transform

        # Extract class names from "class/file.jpg"
        self.samples = []
        for path in file_list:
            class_name = path.split("/")[0]
            self.samples.append((path, class_name))

        # Build class index mapping
        classes = sorted(list({c for _, c in self.samples}))
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.idx_to_class = classes

    def __len__(self):
        return len(self.samples)

    def _resolve_path(self, rel_path):
        p = self.root / rel_path

        # If the path already has an extension, return it
        if p.suffix:
            return p

        # Try common image extensions
        for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
            candidate = p.with_suffix(ext)
            if candidate.exists():
                return candidate

        # If nothing found, raise a clear error
        raise FileNotFoundError(
            f"No valid image file found for '{rel_path}' in {self.root}"
        )

    def __getitem__(self, idx):
        rel_path, class_name = self.samples[idx]
        img_path = self._resolve_path(rel_path)

        img = Image.open(img_path).convert("RGB")
        label = self.class_to_idx[class_name]

        if self.transform:
            img = self.transform(img)

        return img, label


def create_dataloaders(
    train_transform=transforms.Compose,
    test_transform=transforms.Compose,
    data_dir: str = None,
    train_dir: str = None,
    test_dir: str = None,
    train_file: str = None,
    test_file: str = None,
    split: float = 0.8,
    balance: bool = False,
    seed: int = 42,
    batch_size: int = 32,
    num_workers: int = NUM_WORKERS,
):
    sampler = None

    # Pre-existing training/testing directories
    if train_dir and test_dir:
        train_dataset = ImageFolder(train_dir, transform=train_transform)
        test_dataset = ImageFolder(
            test_dir, transform=test_transform or train_transform
        )
        class_names = train_dataset.classes

    # Train/test split from given file
    elif train_file and test_file:
        # print(f"Train_file: {train_file}")
        # print(f"Type: {type(train_file)}")
        if train_file.suffix.lower() in [".txt"]:
            # if train_file.endswith(".txt"):
            with open(train_file, "r") as f:
                train_files = [line.strip() for line in f.readlines()]
        elif train_file.suffix.lower() in [".json"]:
            # elif train_file.endswith(".json"):
            with open(train_file, "r") as f:
                file_data = json.load(f)
            train_files = [item for items in file_data.values() for item in items]
        else:
            raise ValueError(
                f"Unsupported file type for train_file: {train_file} type: {type(train_file)}"
            )
        # print(f"Test_file: {test_file}")
        # print(f"Type: {type(test_file)}")
        if test_file.suffix.lower() in [".txt"]:
            # if test_file.endswith(".txt"):
            with open(test_file, "r") as f:
                test_files = [line.strip() for line in f.readlines()]
        elif test_file.suffix.lower() in [".json"]:
            # elif test_file.endswith(".json"):
            with open(test_file, "r") as f:
                file_data = json.load(f)
            test_files = [item for items in file_data.values() for item in items]
        else:
            raise ValueError(
                f"Unsupported file type for test_file: {test_file} type: {type(test_file)}"
            )
        data_dir = data_dir or Path(train_file).parent / "Images"
        train_dataset = ListToDataset(data_dir, train_files, transform=train_transform)
        test_dataset = ListToDataset(data_dir, test_files, transform=test_transform)
        class_names = train_dataset.idx_to_class

    # Unsplit Dataset from directory, given split ratio
    # Use "balance" flag to accomodate minority class imbalance to 50/50 sampling
    elif data_dir:
        raw_dataset = ImageFolder(data_dir, transform=None)
        class_names = raw_dataset.classes

        labels = np.array([label for _, label in raw_dataset.samples])

        train_idx, test_idx = train_test_split(
            np.arange(len(raw_dataset)),
            test_size=1 - split,
            random_state=seed,
            stratify=labels,
        )
        train_dataset = Subset(
            ImageFolder(data_dir, transform=train_transform), train_idx
        )
        test_dataset = Subset(ImageFolder(data_dir, transform=test_transform), test_idx)

        if balance:
            train_labels = labels[train_idx]
            class_counts = np.bincount(train_labels)

            class_weights = 1.0 / class_counts
            sample_weights = class_weights[train_labels]

            sampler = WeightedRandomSampler(
                weights=torch.DoubleTensor(sample_weights),
                num_samples=len(sample_weights),
                replacement=True,
            )
    else:
        raise ValueError("Invalid Dataloader Input.")

    # Temporarily shrink dataloader for testing
    shrink = 0.5  # percentage reduction
    max_samples = 10000
    if len(train_dataset) > max_samples:
        print(
            f"Using subset of dataset for testing: {shrink * 100:.0f}% of {len(train_dataset)}..."
        )
        keep = int(min(max_samples, len(train_dataset) * shrink))
        train_dataset = Subset(train_dataset, range(keep))
        keep_test = int(min(max_samples, len(test_dataset) * shrink))
        test_dataset = Subset(test_dataset, range(keep_test))

    train_dataloader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=(sampler is None),
        sampler=sampler,
        num_workers=num_workers,
        pin_memory=False,  # True for GPU
    )

    test_dataloader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=False,  # True for GPU
    )

    return train_dataloader, test_dataloader, class_names


# Function for data retrieval from URL source
def download_data(source: str, dest: str, clean: bool = False) -> Path:
    print(f"[START] Beginning data retrieval from {source} & saving to {dest}...")

    # Setup data folder path
    # data_path = Path("data/")
    data_path = Path.cwd().parent / "data/"
    dest_path = data_path / dest

    # Create if data directory does not exist
    if dest_path.is_dir():
        print(f"[COMPLETE] {dest_path} directory exists, skipping retrieval process...")
        print(
            f"[NOTE] If clean data is required, first remove the {dest_path} directory, then re-run this function."
        )
        return dest_path

    print(f"\t[INFO] Creating {dest_path} directory...")
    dest_path.mkdir(parents=True, exist_ok=True)

    response = requests.get(source, allow_redirects=True, stream=True)
    response.raise_for_status()

    cd = response.headers.get("content-disposition")
    if cd and "filename=" in cd:
        filename = cd.split("filename=")[1].strip('"')
    else:
        filename = Path(source.split("?")[0]).name

    file_path = data_path / filename
    print(f"[INFO] Downloading to {file_path}...")

    # Download data
    progress_bar = tqdm(
        total=int(response.headers.get("content-length", 0)),
        unit="iB",
        unit_scale=True,
        leave=True,
        position=0,
    )
    with open(file_path, "wb") as f:
        for data in response.iter_content(1024):
            progress_bar.update(len(data))
            f.write(data)
    progress_bar.close()

    # Unzip
    if filename.endswith(".zip"):
        print("[INFO] Extracting ZIP...")
        with zipfile.ZipFile(file_path, "r") as z:
            z.extractall(dest_path)
    elif (
        filename.endswith(".tar")
        or filename.endswith(".tar.gz")
        or filename.endswith(".tgz")
    ):
        print("[INFO] Extracting TAR...")
        with tarfile.open(file_path, "r:*") as t:
            t.extractall(dest_path)
    else:
        raise ValueError(f"Unsupported archive type: {filename}")

    # Remove .zip file (Optional)
    if clean:
        os.remove(file_path)

    print(f"[COMPLETE] Retrieval process complete, data saved to {dest_path}.")

    return dest_path

In [ ]:
%%writefile {model_builder_script}
"""
Scriptfile for various PyTorch models classes
"""
# Imports
import torch
from torch import nn


class CustomModel(nn.Module):
    """Create a generic model class"""

    def __init__(
        self,
    ) -> None:
        super().__init__()

    def forward(self, x: torch.Tensor):
        return x


class TinyVGG(nn.Module):
    """Creates the TinyVGG architecture.

    Replicates the TinyVGG architecture from the CNN explainer website in PyTorch.
    See the original architecture here: https://poloclub.github.io/cnn-explainer/

    Args:
    input_shape: An integer indicating number of input channels.
    hidden_units: An integer indicating number of hidden units between layers.
    output_shape: An integer indicating number of output units.
    """

    def __init__(self, input_shape: int, hidden_units: int, output_shape: int) -> None:
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(
                in_channels=input_shape,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=0,
            ),
            nn.ReLU(),
            nn.Conv2d(
                in_channels=hidden_units,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=0,
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=0),
            nn.ReLU(),
            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=0),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            # Where did this in_features shape come from?
            # It's because each layer of our network compresses and changes the shape of our inputs data.
            nn.Linear(in_features=hidden_units * 13 * 13, out_features=output_shape),
        )

    def forward(self, x: torch.Tensor):
        x = self.conv_block_1(x)
        x = self.conv_block_2(x)
        x = self.classifier(x)
        return x
        # return self.classifier(self.block_2(self.block_1(x))) # <- leverage the benefits of operator fusion

In [ ]:
%%writefile {model_utils_scripts}
"""
Scriptfile for model utility methods
"""
# Imports
import torch
import torchvision
from torch import nn
from torchinfo import summary

import os
import re
from pathlib import Path

import utils


# Create Efficient Net feature extractor
def create_effnet(model_name: str, num_classes: int = 3, device: str = None):
    print(f"[MODEL] Creating new {model_name} model.")
    utils.reset_seed()

    # Retrieve Effnet Suffix for weights/transforms (_B vs _V2_[S/M/L])
    get_suffix = lambda name: (
        "_".join(name.split("_")[-2:])
        if name.split("_")[-2] == "v2"
        else name.split("_")[-1]
    )
    m_suffix = get_suffix(model_name)
    m_name = f"efficientnet_{m_suffix.lower()}"
    m_weights_name = f"EfficientNet_{m_suffix.upper()}_Weights"
    weights = getattr(torchvision.models, m_weights_name).DEFAULT
    transforms = weights.transforms()
    model = getattr(torchvision.models, m_name)(weights=weights).to(device)

    # Freeze base layers
    for param in model.features.parameters():
        param.requires_grad = False

    # Retrieve output layer classifier settings
    classifier = model.classifier
    dropout = classifier[0].p if isinstance(classifier[0], nn.Dropout) else 0.0
    in_features = classifier[-1].in_features

    # Adjust the output layer based on model type
    utils.reset_seed()
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features=in_features, out_features=num_classes),
    ).to(device)

    model.name = model_name
    print(
        f"[MODEL] {model.name} created:"
        f"\n\t[Weights]: {m_weights_name}"
        f"\n\t[Dropout]: {dropout}"
        f"\n\t[In Features]: {in_features}"
        f"\n\t[Out Features]: {num_classes}"
    )

    # # Get model summary (uncomment for full output)
    # model_summary = summary(
    #     model,
    #     input_size=(1, 3, 224, 224),
    #     col_names=["input_size", "output_size", "num_params", "trainable"],
    #     col_width=20,
    #     row_settings=["var_names"],
    #     verbose=0,  # Comment out for output
    # )
    # # Write the model summary to file
    # with open(f"{model.name}_summary.txt", "w", encoding="utf-8") as f:
    #     f.write(str(model_summary))

    return model, transforms


# Create Res Net feature extractor
def create_resnet(model_name: str, num_classes: int = 3, device: str = None):
    print(f"[MODEL] Creating new {model_name} model.")
    utils.reset_seed()

    # Retrieve ResNet Suffix/Versions for weights/transforms
    m_vers = re.search(r"resnet(\d+)", model_name).group(1)
    m_name = f"resnet{m_vers}"
    is_v2 = "_v2" in model_name
    m_weights_name = f"ResNet{m_vers}_Weights"
    weights_enum = getattr(torchvision.models, m_weights_name)
    if is_v2:
        weights = weights_enum.IMAGENET1K_V2
        m_weights_name = f"ResNet{m_vers}_Weights.IMAGENET1K_V2"
    else:
        weights = weights_enum.DEFAULT
        m_weights_name = f"ResNet{m_vers}_Weights.IMAGENET1K_V1"
    transforms = weights.transforms()
    model = getattr(torchvision.models, m_name)(weights=weights).to(device)

    # Freeze base layers
    for param in model.parameters():
        param.requires_grad = False

    # Adjust the output layer based on model type
    utils.reset_seed()
    model.fc = nn.Linear(in_features=model.fc.in_features, out_features=num_classes).to(
        device
    )

    model.name = model_name
    print(
        f"[MODEL] {model.name} created:"
        f"\n\t[Weights]: {m_weights_name}"
        f"\n\t[In Features]: {model.fc.in_features}"
        f"\n\t[Out Features]: {num_classes}"
    )

    # # Get model summary (uncomment for full output)
    # model_summary = summary(
    #     model,
    #     input_size=(1, 3, 224, 224),
    #     col_names=["input_size", "output_size", "num_params", "trainable"],
    #     col_width=20,
    #     row_settings=["var_names"],
    #     verbose=0,  # Comment out for output
    # )
    # # Write the model summary to file
    # with open(f"{model.name}_summary.txt", "w", encoding="utf-8") as f:
    #     f.write(str(model_summary))

    return model, transforms


# Create VGG feature extractor
def create_vgg(model_name: str, num_classes: int = 3, device: str = None):
    print(f"[MODEL] Creating new {model_name} model.")
    utils.reset_seed()

    # Retrieve VGG Suffix/Versions for weights/transforms
    m_vers = re.search(r"vgg(\d+)", model_name).group(1)
    m_name = f"vgg{m_vers}"
    m_weights_name = f"VGG{m_vers}_Weights"
    weights = getattr(torchvision.models, m_weights_name).DEFAULT
    transforms = weights.transforms()
    model = getattr(torchvision.models, m_name)(weights=weights).to(device)

    # Freeze base layers
    for param in model.parameters():
        param.requires_grad = False

    # Adjust the output layer based on model type
    utils.reset_seed()
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes).to(device)

    model.name = model_name
    print(
        f"[MODEL] {model.name} created:"
        f"\n\t[Weights]: {m_weights_name}"
        f"\n\t[In Features]: {in_features}"
        f"\n\t[Out Features]: {num_classes}"
    )

    # # Get model summary (uncomment for full output)
    # model_summary = summary(
    #     model,
    #     input_size=(1, 3, 224, 224),
    #     col_names=["input_size", "output_size", "num_params", "trainable"],
    #     col_width=20,
    #     row_settings=["var_names"],
    #     verbose=0,  # Comment out for output
    # )
    # # Write the model summary to file
    # with open(f"{model.name}_summary.txt", "w", encoding="utf-8") as f:
    #     f.write(str(model_summary))

    return model, transforms


def create_vit_model(num_classes: int = 3, device: str = None):
    weights = torchvision.models.ViT_B_16_Weights.DEFAULT
    transforms = weights.transforms()
    model = torchvision.models.vit_b_16(weights=weights)

    for param in model.parameters():
        param.requires_grad = False

    # Adjust the output layer as necessary
    utils.reset_seed()
    model.heads = nn.Sequential(
        nn.Linear(
            in_features=768,
            out_features=num_classes,
        )
    ).to(device)

    model.name = "ViT"
    print(f"[INFO] Creating new {model.name} model.")

    # # Print ViT feature extractor model summary (uncomment for full output)
    # summary(vit,
    #         input_size=(1, 3, 224, 224),
    #         col_names=["input_size", "output_size", "num_params", "trainable"],
    #         col_width=20,
    #         row_settings=["var_names"])

    return model, transforms


# Function for loading a PyTorch Model checkpoint
def load_checkpoint(
    model_name: str,
    checkpoint_dir: str,
    checkpoint_name: str,
) -> dict:
    checkpoint_path = Path(checkpoint_dir) / f"{checkpoint_name}.tar"

    print(f"Checkpoint Path: {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path)

    return checkpoint


# Function for saving PyTorch Model checkpoint
def save_checkpoint(
    state: dict,
    checkpoint_dir: str,
    checkpoint_name: str,
):
    """
    Save checkpoint to target directory.
    """
    # Create target directory
    target_dir_path = Path(checkpoint_dir)
    target_dir_path.mkdir(parents=True, exist_ok=True)

    checkpoint_file = target_dir_path / f"{checkpoint_name}_Epoch{state['epoch']}.tar"
    if checkpoint_file.exists():
        counter = 1
        while True:
            # print(f"[INFO] {checkpoint_file} file exists, retrying... ")
            checkpoint_file_new = (
                target_dir_path
                / f"{checkpoint_name}_Counter{counter}_Epoch{state['epoch']}.tar"
            )
            checkpoint_file = checkpoint_file_new
            if not checkpoint_file.exists():
                break
            counter += 1

    # checkpoint_file = checkpoint_file.with_suffix(".tar")
    print(f"[INFO] Saving checkpoint to: {checkpoint_file}")
    torch.save(state, checkpoint_file)

    return


# Function for saving PyTorch model to target directory
def save_model(model: torch.nn.Module, target_dir: str, model_filename: str):
    """Saves a single PyTorch model to a target directory.

    Args:
        model:      A target PyTorch model to save.
        target_dir: A directory for saving the model to.
        model_name: A filename for the saved model extension.

    Example usage:
        save_model(model=model_0,
                   target_dir="models",
                   model_name="05_going_modular_tingvgg_model")
    """
    # Create target directory
    target_dir_path = Path(target_dir)
    target_dir_path.mkdir(parents=True, exist_ok=True)

    model_file = target_dir_path / f"{model_filename}.pt"
    if model_file.exists():
        counter = 1
        while True:
            # print(f"[INFO] {model_file} file exists, retrying... ")
            model_file_new = target_dir_path / f"{model_filename}_Dup{counter}.pt"
            model_file = model_file_new
            if not model_file.exists():
                break
            counter += 1

    # model_file = model_file.with_suffix(".pt")
    print(f"[INFO] Saving model to: {model_file}")
    torch.save(obj=model.state_dict(), f=model_file)

    return

In [ ]:
%%writefile {engine_script}
"""
Scriptfile for training & testing PyTorch models
Tracks loss & accuracy by default, adjust implementation for additional metrics as necessary
"""
# Imports
from typing import Dict, List, Tuple
import torch
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from datetime import datetime

import model_utils


# Add writer parameter to train()
def train(
    model_name: str,
    model: torch.nn.Module,
    dataset_name: str,
    train_dataloader: torch.utils.data.DataLoader,
    test_dataloader: torch.utils.data.DataLoader,
    loss_fn: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    epochs: int,
    device: torch.device,
    writer: torch.utils.tensorboard.writer.SummaryWriter,
    checkpoint: bool,
    checkpoint_name: str,
) -> Dict[str, List]:
    """Trains and tests a PyTorch model.

    Passes a target PyTorch models through train_step() and test_step() functions for a number of epochs, training and testing the model in the same epoch loop.

    Calculates, prints and stores evaluation metrics throughout, & stores metrics to specified writer log_dir if present.

    Args:
        model:              A PyTorch model to be trained and tested.
        train_dataloader:   A DataLoader instance for the model to be trained on.
        test_dataloader:    A DataLoader instance for the model to be tested on.
        optimizer:          A PyTorch optimizer to help minimize the loss function.
        loss_fn:            A PyTorch loss function to calculate loss on both datasets.
        epochs:             An integer indicating how many epochs to train for.
        device:             A target device to compute on (e.g. "cuda" or "cpu").
        writer:             A SummaryWriter() instance to log model results to.

    Returns:
        A dictionary of training and testing loss as well as training and testing accuracy metrics. Each metric has a value in a list for each epoch.
        In the form:
            {train_loss: [...],
             train_acc: [...],
             test_loss: [...],
             test_acc: [...]}
        For example if training for epochs=2:
            {train_loss: [2.0616, 1.0537],
             train_acc: [0.3945, 0.3945],
             test_loss: [1.2641, 1.5706],
             test_acc: [0.3400, 0.2973]}
    """
    # Create empty metrics results dictionary
    results = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

    # Loop through training and testing steps for a number of epochs
    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(
            model=model,
            dataloader=train_dataloader,
            loss_fn=loss_fn,
            optimizer=optimizer,
            device=device,
        )

        test_loss, test_acc = test_step(
            model=model, dataloader=test_dataloader, loss_fn=loss_fn, device=device
        )

        # Print out what's happening
        looptime = datetime.now().strftime("%m-%d %H:%M:%S")
        print(
            f"[{looptime}]"
            f"Epoch: {epoch + 1} | "
            f"train_loss: {train_loss:.4f} | "
            f"test_loss: {test_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"test_acc: {test_acc:.4f}"
        )

        # Update results dictionary
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

        # Flag if saving checkpoints
        if checkpoint:
            checkpoint_state = {
                "model_name": model_name,
                "epoch": epoch,
                "next_epoch": epoch + 1,
                "num_epochs": epochs,
                "state_dict": model.state_dict(),
                "loss_fn": loss_fn.state_dict(),
                "optimizer": optimizer.state_dict(),
                "device": device,
                "results": results,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "test_loss": test_loss,
                "test_acc": test_acc,
            }
            model_utils.save_checkpoint(
                state=checkpoint_state,
                checkpoint_dir=f"models/{model_name}/{dataset_name}/checkpoints",
                checkpoint_name=f"{checkpoint_name}",
            )
        # Use the writer parameter to track experiments if exists
        if writer:
            # Add results to SummaryWriter
            writer.add_scalars(
                main_tag="Loss",
                tag_scalar_dict={"train_loss": train_loss, "test_loss": test_loss},
                global_step=epoch,
            )
            writer.add_scalars(
                main_tag="Accuracy",
                tag_scalar_dict={"train_acc": train_acc, "test_acc": test_acc},
                global_step=epoch,
            )

            # Close the writer
            writer.close()
        else:
            pass

    # Close the writer
    # if writer:
    #     writer.close()

    # Return the filled results at the end of the epochs
    return results


def train_step(
    model: torch.nn.Module,
    dataloader: torch.utils.data.DataLoader,
    loss_fn: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> Tuple[float, float]:
    """Trains a PyTorch model for a single epoch.

    Turns a target PyTorch model to training mode and then runs through all of the required training steps (forward pass, loss calculation, optimizer step).

    Args:
        model:      A PyTorch model to be trained.
        dataloader: A DataLoader instance for the model to be trained on.
        loss_fn:    A PyTorch loss function to minimize.
        optimizer:  A PyTorch optimizer to help minimize the loss function.
        device:     A target device to compute on (e.g. "cuda" or "cpu").

    Returns:
        A tuple of training loss and training accuracy metrics, in the form (train_loss, train_accuracy).
            e.g.:   (0.1112, 0.8743)
    """
    # Put model in train mode
    model.train()

    # Initialize train loss and train accuracy values
    train_loss, train_acc = 0, 0

    # Loop through data loader data batches
    for batch, (X, y) in enumerate(dataloader):
        # Send data to target device
        X, y = X.to(device), y.to(device)

        # 1. Forward pass
        y_pred = model(X)

        # 2. Calculate  and accumulate loss
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()

        # 3. Optimizer zero grad
        optimizer.zero_grad()

        # 4. Loss backward
        loss.backward()

        # 5. Optimizer step
        optimizer.step()

        # Calculate and accumulate accuracy metric across all batches
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item() / len(y_pred)

    # Adjust metrics to get average loss and accuracy per batch
    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)

    return train_loss, train_acc


def test_step(
    model: torch.nn.Module,
    dataloader: torch.utils.data.DataLoader,
    loss_fn: torch.nn.Module,
    device: torch.device,
) -> Tuple[float, float]:
    """Tests a PyTorch model for a single epoch.

    Turns a target PyTorch model to "eval" mode and then performs a forward pass on a testing dataset.

    Args:
        model:      A PyTorch model to be tested.
        dataloader: A DataLoader instance for the model to be tested on.
        loss_fn:    A PyTorch loss function to calculate loss on the test data.
        device:     A target device to compute on (e.g. "cuda" or "cpu").

    Returns:
        A tuple of testing loss and testing accuracy metrics, in the form (test_loss, test_accuracy).
            e.g.    (0.0223, 0.8985)
    """
    # Put model in eval mode
    model.eval()

    # Setup test loss and test accuracy values
    test_loss, test_acc = 0, 0

    # Turn on inference context manager
    with torch.inference_mode():
        # Loop through DataLoader batches
        for batch, (X, y) in enumerate(dataloader):
            # Send data to target device
            X, y = X.to(device), y.to(device)

            # 1. Forward pass
            test_pred_logits = model(X)

            # 2. Calculate and accumulate loss
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()

            # Calculate and accumulate accuracy
            test_pred_labels = test_pred_logits.argmax(dim=1)
            test_acc += (test_pred_labels == y).sum().item() / len(test_pred_labels)

    # Adjust metrics to get average loss and accuracy per batch
    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss, test_acc

In [ ]:
%%writefile {train_script}
"""
Scriptfile for executing model training loop evaluation.
"""
# Imports
import os
import sys
import json
import time
import requests
import urllib.request
import platform
import argparse
import subprocess
from pathlib import Path

import torch
import torch.multiprocessing as mp
from torchvision import transforms, datasets
from torchinfo import summary

import data_fn, engine, model_builder, model_utils, stats, utils, install_requirements


def execute_training_loop(
    pcounter: int,
    nprocs: int,
    model_name: str,
    model: torch.nn.Module,
    dataset_name: str,
    train_dataloader: torch.utils.data.DataLoader,
    test_dataloader: torch.utils.data.DataLoader,
    loss_fn: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    epochs: int,
    device: torch.device,
    checkpoint: bool,
    dkey: str,
    mkey: str,
):
    print(f"[START] Executing Process {pcounter}")

    # Execute model training
    model_results = engine.train(
        model_name=model_name,
        model=model,
        dataset_name=dataset_name,
        train_dataloader=train_dataloader,
        test_dataloader=test_dataloader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        epochs=epochs,
        device=device,
        writer=utils.create_writer(
            experiment_name=dataset_name,
            model_name=model_name,
            extra=mkey,
        ),
        checkpoint=checkpoint,
        checkpoint_name=mkey,
    )

    model_utils.save_model(
        model=model,
        target_dir=f"models/{model_name}/{dataset_name}",
        model_filename=mkey,
    )
    print("-" * 50 + "\n")

    stats.save_results(
        target_dir=f"results/{dataset_name}",
        model_name=model_name,
        results=model_results,
    )

    stats.plot_loss_curves(model_results)

    print(f"[END] Process {pcounter} complete")

    return


def main():
    # Setup Arg Parser parameters (Change defaults as necessary)
    package_name = "Translation Model Package"

    # Install packages if necessary
    print(f"[INFO] Installing necessary packages...")
    install_requirements.install_external_imports()

    class CustomFormatter(
        argparse.ArgumentDefaultsHelpFormatter, argparse.RawTextHelpFormatter
    ):
        pass

    parser = argparse.ArgumentParser(
        # formatter_class=CustomFormatter,
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
        description=f"{package_name} Training Script.",
    )

    parser.add_argument(
        "--DATA_DL",
        type=str,
        default=None,
        help="Indicate URL source to attempt full dataset download",
    )
    parser.add_argument(
        "--LOAD_CHECKPOINT",
        type=str,
        default=None,
        help="Indicate relative path if loading checkpoint file",
    )
    parser.add_argument(
        "--DATA_DIR",
        type=str,
        default="data",
        help="Default Data directory, training/testing split to be generated within program.  Assumes relative location at the parent level: (e.g.)\n\t/{package_name}/train.py\n\t/data/{data_dir}\n\t",
    )
    parser.add_argument(
        "--TRAIN_DIR",
        nargs="+",
        type=str,
        default="pizza_steak_sushi/train",
        help="Training data directory location",
    )
    parser.add_argument(
        "--TEST_DIR",
        nargs="+",
        type=str,
        default="pizza_steak_sushi/test",
        help="Testing data directory location",
    )
    parser.add_argument(
        "--NUM_CLASSES",
        type=int,
        default=3,
        help="Number of classes in the dataset",
    )
    parser.add_argument(
        "--MODEL_NAME",
        nargs="+",
        type=str,
        default=["effnetb0", "effnetb2", "CustomModel"],
        help="Model name",
    )
    parser.add_argument(
        "--NUM_EPOCHS",
        nargs="+",
        type=int,
        default=[5, 10],
        help="Epoch Loops",
    )
    parser.add_argument(
        "--BATCH_SIZE",
        nargs="+",
        type=int,
        default=[1, 32],
        help="Batch size",
    )
    parser.add_argument(
        "--LEARNING_RATE",
        nargs="+",
        type=float,
        default=[1e-5, 1e-3, 1e-1],
        help="Training Learning rate",
    )
    parser.add_argument(
        "--DDP_FLAG",
        type=bool,
        default=False,
        help="Indicates if runningg process through DDP",
    )
    parser.add_argument(
        "--TEST",
        type=bool,
        default=False,
        help="Run tests",
    )
    parser.add_argument(
        "--DOWNLOAD",
        type=bool,
        default=True,
        help="Download datasets",
    )

    # Argparser storage
    args = parser.parse_args()

    if args.TEST:
        print(f"Test mode enabled")
        testing = True
        test_dataset = True
        # test_setup = True
    else:
        testing = True
        test_dataset = False
        test_setup = False

    # Get latest LOL version & download loading screen splashes
    if args.DOWNLOAD:
        # data_fn.download_datasets(data_dir=Path.cwd().parent / "data/")
        data_fn.download_lol_dataset(data_dir=Path.cwd().parent / "data/lol_champions")

    # Set default args for simpler testing
    if locals().get("testing", True):
        # # All EfficientNet models
        # args.MODEL_NAME = [
        #     "effnet_b0",
        #     "effnet_b1",
        #     "effnet_b2",
        #     "effnet_b3",
        #     "effnet_b4",
        #     "effnet_b5",
        #     "effnet_b6",
        #     "effnet_v2_s",
        #     "effnet_v2_m",
        #     "effnet_v2_l",
        # ]
        # # All Resenet models
        # args.MODEL_NAME = [
        #     "resnet18",
        #     "resnet34",
        #     "resnet50",
        #     "resnet50_V2",
        #     "resnet101",
        #     "resnet101_V2",
        #     "resnet152",
        #     "resnet152_V2",
        # ]
        # # All VGG models
        # args.MODEL_NAME = [
        #     "vgg11",
        #     "vgg11_BN",
        #     "vgg13",
        #     "vgg13_BN",
        #     "vgg16",
        #     "vgg16_BN",
        #     "vgg19",
        #     "vgg19_BN",
        # ]
        args.MODEL_NAME = ["effnet_b0", "effnet_b2", "resnet18", "vgg11"]
        args.NUM_EPOCHS = [5]
        args.BATCH_SIZE = [16]
        args.LEARNING_RATE = [1e-5]

        # Setup default data directory (update relative locations)
        data_dir = Path.cwd().parent / "data/"

        # Download datasets
        # data_fn.download_datasets(data_dir)
        data_fn.download_lol_dataset(data_dir=data_dir / "lol_champions")

        DATA_DICT = {}

        # Using LOL champion dataset
        DATA_DICT[len(DATA_DICT)] = {
            "train_dir": data_dir / "lol_champions",
            "test_dir": data_dir / "lol_champions",
            "class_names": datasets.ImageFolder(
                data_dir / "lol_champions", transform=None
            ).classes,
        }

    else:
        data_dir = Path(args.DATA_DIR)
        DATA_DICT = {}
        for idx, dir in enumerate(args.TRAIN_DIR):
            train_path = data_dir / dir
            if dir.endswith(".txt"):
                class_names = sorted(
                    {line.strip().split("/")[0] for line in open(train_path, "r")}
                )
            elif train_file.endswith(".json"):
                file_data = json.load(open(train_path, "r"))
                if isinstance(file_data, list):
                    class_names = sorted({item.split("/")[0] for item in file_data})
                elif isinstance(file_data, dict) and "train" in file_data:
                    class_names = sorted(
                        {item.split("/")[0] for item in file_data["train"]}
                    )
                else:
                    class_names = sorted(file_data.keys())
            else:
                class_names = datasets.ImageFolder(
                    data_dir / dir, transform=None
                ).classes
            DATA_DICT[len(DATA_DICT)] = {
                "train_dir": data_dir / dir,
                "test_dir": data_dir / args.TEST_DIR[idx],
                "class_names": class_names,
            }

    # # Set workers (if applicable)
    # if platform.system() == "Windows":
    #     NUM_WORKERS = 0
    # else:
    #     # NUM_WORKERS = os.cpu_count()
    #     NUM_WORKERS = 0

    # Set device & MP Params
    if torch.cuda.is_available():
        device = "cuda"

        if platform.system() == "Windows":
            NUM_WORKERS = 0
        else:
            NUM_WORKERS = os.cpu_count()

        # MP parameters
        mp.set_start_method("spawn", force=True)
        world_size = max(1, mp.cpu_count() - 1)
        max_workers = max(1, mp.cpu_count() - 1)  # Keep one CPU free
        if locals().get("test_dataset", True):
            world_size = 1
            max_workers = 1
            print(f"[TEST] Testing Large dataset, max_workers set to {max_workers}...")

        # Increase hyperparameters based on GPU
        args.NUM_EPOCHS[0] = 100  # X10
        args.BATCH_SIZE[0] = 32  # X2
        args.LEARNING_RATE = [1e-6]
    else:
        device = "cpu"
        NUM_WORKERS = 0

        # Non-distributed Multiprocessing settings
        max_workers = max(1, mp.cpu_count() - 1)  # Keep one CPU free
        if locals().get("test_dataset", True):
            max_workers = 2  # Set smaller number for testing
            print(f"[TEST] Testing Large dataset, max_workers set to {max_workers}...")

    # Multiprocessing objects
    active_processes = []
    pcounter = 0

    # Multi-experiment trackers
    EXPERIMENT_NUMBER = 0
    utils.reset_seed()

    # Initialize reference param tracking
    reference_params = {}
    reference_params["models"] = {}
    reference_params["datasets"] = {}

    # Instantiate Models
    ML_DICT = {}
    model_transforms = {}  # Loop through feature extractors and save pairs
    ml_counter = 0
    for model_name in args.MODEL_NAME:
        for idx, value in DATA_DICT.items():
            class_names = value["class_names"]
            for epochs in args.NUM_EPOCHS:
                for lr in args.LEARNING_RATE:
                    ml_counter += 1
                    ml_key = f"{model_name}_Model_{ml_counter}"
                    print(f"[MODEL] Creating {ml_key} model...")
                    utils.reset_seed()
                    # Instantiate default model
                    model = model_builder.TinyVGG(
                        input_shape=3,
                        hidden_units=10,
                        output_shape=len(class_names),
                    ).to(device)
                    if "effnet" in model_name.lower():
                        model, model_transforms[model_name] = model_utils.create_effnet(
                            model_name=model_name.lower(),
                            num_classes=len(class_names),
                            device=device,
                        )
                    if "resnet" in model_name.lower():
                        model, model_transforms[model_name] = model_utils.create_resnet(
                            model_name=model_name.lower(),
                            num_classes=len(class_names),
                            device=device,
                        )
                    if "vgg" in model_name.lower():
                        model, model_transforms[model_name] = model_utils.create_vgg(
                            model_name=model_name.lower(),
                            num_classes=len(class_names),
                            device=device,
                        )
                    # Set loss function & optimizer
                    loss_fn = torch.nn.CrossEntropyLoss()
                    optimizer = torch.optim.Adam(params=model.parameters(), lr=lr)

                    ML_DICT[ml_key] = {
                        "model_name": model_name,
                        "num_class": len(class_names),
                        "model": model,
                        "num_epochs": epochs,
                        "learning_rate": lr,
                        "loss_fn": loss_fn,
                        "optimizer": optimizer,
                        "device": device,
                    }
                    reference_params["models"][ml_key] = {
                        "model_name": model_name,
                        "num_class": len(class_names),
                        "loss_fn": str(loss_fn),
                        "optimizer": str(optimizer).split(" ")[0],
                        "num_epochs": epochs,
                        "learning_rate": lr,
                        "device": device,
                    }

    # Add custom transformations if testing (Uncomment if necessary)
    """ 
    # Setup normalizations (ImageNet distribution used below)
    normalize = transforms.Normalize(
        mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
    )
    custom_transform = transforms.Compose(
        [transforms.Resize((64, 64)), transforms.ToTensor(), normalize]
    )
    # Change key/value based on Custom designations
    model_transforms["CustomModel"] = custom_transform
    """

    # Instantiate Dataloaders
    DL_DICT = {}
    dl_counter = 0
    dir_counter = 0
    for idx, value in DATA_DICT.items():
        train_dir = value["train_dir"]
        test_dir = value["test_dir"]
        class_names = value["class_names"]
        if train_dir == test_dir:
            dataset_name = f"{value['train_dir'].name}"
        else:
            dataset_name = f"{value['train_dir'].parent.name}"
        print(
            f"[DATA] Dataset: {dataset_name}\nTrain_dir: {train_dir}\nTest_dir: {test_dir}"
        )
        # for train_dir in TRAIN_DIR_LIST:
        #     dataset_name = f"{train_dir.parent.name}"
        #     test_dir = TEST_DIR_LIST[dir_counter]
        #     dir_counter += 1
        # print(f"Train_dir: {train_dir}")
        # print(f"Test_dir: {test_dir}")
        # for test_dir in TEST_DIR_LIST:
        #     print(f"Test_dir: {test_dir}")
        #     if dataset_name != f"{test_dir.parent.name}":
        #         print(f"[WARNING] Directories do not match, skipping...")
        #         continue
        # for tname, transform in TRANSFORMATIONS.items(): # Custom Transforms
        for model_name, model_transform in model_transforms.items():
            transform = transforms.Compose(
                [transforms.Grayscale(num_output_channels=3), model_transform]
            )
            for bs in args.BATCH_SIZE:
                dl_counter += 1
                dl_key = f"{dataset_name}_Dataset_{dl_counter}"
                print(f"[DATA] Creating {dl_key} Dataloader...")
                utils.reset_seed()

                if train_dir.is_dir() and "train" in train_dir.name.lower():
                    # print(f"Directory mode:")
                    train_dataloader, test_dataloader, num_classes = (
                        data_fn.create_dataloaders(
                            train_dir=train_dir,
                            test_dir=test_dir,
                            train_transform=transform,
                            test_transform=transform,
                            batch_size=bs,
                            num_workers=NUM_WORKERS,
                        )
                    )
                    data_fn.check_dataloader(train_dir, train_dataloader)
                    data_fn.check_dataloader(test_dir, test_dataloader)
                elif train_dir.suffix.lower() in [".txt", ".json"]:
                    # print(f"File-list mode:")
                    train_dataloader, test_dataloader, num_classes = (
                        data_fn.create_dataloaders(
                            # data_dir=data_dir,
                            train_file=train_dir,
                            test_file=test_dir,
                            train_transform=transform,
                            test_transform=transform,
                            batch_size=bs,
                            num_workers=NUM_WORKERS,
                        )
                    )
                    data_fn.check_dataloader(train_dir, train_dataloader)
                    data_fn.check_dataloader(test_dir, test_dataloader)
                else:
                    # print(f"Unsplit mode:")
                    train_dataloader, test_dataloader, num_classes = (
                        data_fn.create_dataloaders(
                            data_dir=train_dir,
                            split=0.8,
                            balance=False,
                            train_transform=transform,
                            test_transform=transform,
                            batch_size=bs,
                            num_workers=NUM_WORKERS,
                        )
                    )
                    data_fn.check_dataloader(train_dir, train_dataloader)
                    data_fn.check_dataloader(test_dir, test_dataloader)

                # train_dataloader, test_dataloader, class_names = (
                #     data_fn.create_dataloaders(
                #         train_dir=train_dir,
                #         test_dir=test_dir,
                #         train_transform=transform,
                #         test_transform=transform,
                #         batch_size=bs,
                #         num_workers=NUM_WORKERS,
                #     )
                # )
                DL_DICT[dl_key] = {
                    "dataset_name": dataset_name,
                    "model_transform": model_name,
                    "transformation": transform,
                    "batch_size": bs,
                    "num_workers": NUM_WORKERS,
                    "train_dataloader": train_dataloader,
                    "test_dataloader": test_dataloader,
                    "class_names": class_names,
                    "num_classes": len(class_names),
                }
                n = 3
                reference_params["datasets"][dl_key] = {
                    "dataset_name": dataset_name,
                    "model_transform": model_name,
                    "transformation": str(transform),
                    "batch_size": bs,
                    "num_workers": NUM_WORKERS,
                    "class_names": f"Displaying {n} of {len(class_names)}\n\t - {class_names[:n]}",
                    "num_classes": len(class_names),
                }
    # Store class names for model building
    # class_names = DL_DICT[next(iter(DL_DICT))]["class_names"]

    # Write params to reference file
    with open(f"params.json", "w") as json_file:
        json.dump(reference_params, json_file, indent=4)

    # Quit process to verify initial setup (Comment Out)
    if locals().get("test_setup", False):
        print(f"[TEST] Check setup...")
        sys.exit(1)

    # Load checkpoint
    if args.LOAD_CHECKPOINT:
        print(f"[INFO] Attempting to load checkpoint {args.LOAD_CHECKPOINT}...")
        # Instantiate default model
        model = model_builder.TinyVGG(
            input_shape=3,
            hidden_units=10,
            output_shape=len(class_names),
        ).to(device)
        if "effnet" in model_name:
            model, model_transforms[model_name] = model_utils.create_effnet(
                model_name=model_name,
                num_classes=len(class_names),
                device=device,
            )
        checkpoint = model_utils.load_checkpoint(
            checkpoint_dir="checkpoints",
            checkpoint_name=args.LOAD_CHECKPOINT,
        )
        model.load_state_dict(checkpoint["state_dict"])
        loss_fn = torch.nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(params=model.parameters(), lr=lr)
        optimizer.load_state_dict(checkpoint["optimizer"])
    else:
        try:
            nprocs = int(len(DL_DICT)) * int(len(ML_DICT))
            for dkey, dvalue in DL_DICT.items():
                for mkey, mvalue in ML_DICT.items():
                    pcounter += 1
                    if mvalue["model_name"] != dvalue["model_transform"]:
                        # Skip loop pair if current dataset transform does not match the current model
                        print(
                            f"[INFO] Current model does not match transformations applied to current dataset, skipping Process {pcounter}..."
                        )
                        continue
                    print(f"[INFO] Spawning Process {pcounter} of expected {nprocs}...")
                    # print(f"Dkey:\t{dkey}\nDvalue:\n\t{dvalue}")
                    # print(f"Mkey:\t{mkey}\nMvalue:\n\t{mvalue}")
                    model_name = mvalue["model_name"]
                    model = mvalue["model"]
                    dataset_name = dvalue["dataset_name"]
                    train_dataloader = dvalue["train_dataloader"]
                    test_dataloader = dvalue["test_dataloader"]
                    loss_fn = mvalue["loss_fn"]
                    optimizer = mvalue["optimizer"]
                    epochs = mvalue["num_epochs"]
                    device = mvalue["device"]
                    checkpoint = False  # True to save checkpoint files

                    # Execute process per experiement
                    while len(active_processes) >= max_workers:
                        active_processes = [p for p in active_processes if p.is_alive()]
                        time.sleep(0.1)

                    p = mp.Process(
                        target=execute_training_loop,
                        args=(
                            pcounter,
                            nprocs,
                            model_name,
                            model,
                            dataset_name,
                            train_dataloader,
                            test_dataloader,
                            loss_fn,
                            optimizer,
                            epochs,
                            device,
                            checkpoint,
                            dkey,
                            mkey,
                        ),
                    )

                    p.start()
                    active_processes.append(p)

                    # DDP-Style multiprocessing uses mp.spawn
                    # Prepends {rank} in execute_training_loop() if used
                    """
                    mp.spawn(
                        execute_training_loop,
                        args=(
                            world_size,
                            model_name,
                            model,
                            dkey,
                            train_dataloader,
                            test_dataloader,
                            loss_fn,
                            optimizer,
                            epochs,
                            device,
                            checkpoint,
                            dkey,
                            mkey,
                        ),
                        nprocs=world_size,
                        join=True,
                    )
                    """
            for p in active_processes:
                p.join()
            print(f"[INFO] All procceses complete!")
        except Exception as e:
            print(f"[ERROR] Error spawning process {pcounter}: {e}")

    # main(args)
    print("[END] Training complete")

    sys.exit(0)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {stats_script}
"""
Scriptfile for stat capture and visualization
"""
# Imports

import os
import json

import torch
import torchvision
from torchvision import transforms

from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
from typing import List, Tuple

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"


# Save results after model training
def save_results(target_dir: str, model_name: str, results: dict):
    target_path = Path(target_dir)
    target_path.mkdir(parents=True, exist_ok=True)
    results_file = f"{target_dir}/{model_name}"

    if Path(f"{results_file}_results.json").exists():
        counter = 1
        while True:
            print(f"[INFO] {results_file}_results.json file exists, retrying... ")
            results_file_new = f"{target_dir}/{model_name}_{counter}"
            results_file = results_file_new
            if not Path(f"{results_file}_results.json").exists():
                break
            counter += 1

    print(f"[INFO] Writing to {results_file}...")
    with open(f"{results_file}_results.json", "w") as json_file:
        json.dump(results, json_file, indent=4)

    loss_file = f"{results_file}_loss.png"
    results["model_name"] = model_name
    plot_loss_curves(results, loss_file)

    return


# Plot model loss curves
def plot_loss_curves(results: dict, save_path: str = None):
    """Plots training curves of a results dictionary.

    Args:
        results (dict): dictionary containing list of values, e.g.
            {"train_loss": [...],
             "train_acc": [...],
             "test_loss": [...],
             "test_acc": [...]}
    """
    loss = results["train_loss"]
    test_loss = results["test_loss"]

    accuracy = results["train_acc"]
    test_accuracy = results["test_acc"]

    epochs = range(len(results["train_loss"]))

    plt.figure(figsize=(15, 8))

    # Make this the graph title
    if "model_name" in results:
        graph_title = results["model_name"]
        plt.suptitle(graph_title, fontsize=16)

    # Plot loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, loss, label="train_loss")
    plt.plot(epochs, test_loss, label="test_loss")
    plt.title("Loss")
    plt.xlabel("Epochs")
    plt.legend()

    # Plot accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs, accuracy, label="train_accuracy")
    plt.plot(epochs, test_accuracy, label="test_accuracy")
    plt.title("Accuracy")
    plt.xlabel("Epochs")
    plt.legend()

    if save_path:
        print(f"[INFO] Saving Loss curve to {save_path}...")
        plt.savefig(save_path)

    return


# Display an image with accompanying model prediction probabilities
def plot_image_prediction(
    model: torch.nn.Module,
    class_names: List[str],
    image_path: str,
    image_size: Tuple[int, int] = (224, 224),
    transform: torchvision.transforms = None,
    device: torch.device = device,
):
    """Predicts on a target image with a target model.

    Args:
        model (torch.nn.Module): A trained (or untrained) PyTorch model to predict on an image.
        class_names (List[str]): A list of target classes to map predictions to.
        image_path (str): Filepath to target image to predict on.
        image_size (Tuple[int, int], optional): Size to transform target image to. Defaults to (224, 224).
        transform (torchvision.transforms, optional): Transform to perform on image. Defaults to None which uses ImageNet normalization.
        device (torch.device, optional): Target device to perform prediction on. Defaults to device.
    """
    # Create transformation for image (if one doesn't exist)
    default_transform = transforms.Compose(
        [
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )
    if transform is not None:
        image_transform = transform
    else:
        image_transform = default_transform

    # Open image
    img = Image.open(image_path)

    # Make sure the model is on the target device
    model.to(device)

    # Turn on model evaluation mode and inference mode
    model.eval()
    with torch.inference_mode():
        # Transform and add an extra dimension to image (model requires samples in [batch_size, color_channels, height, width])
        transformed_image = image_transform(img).unsqueeze(dim=0)
        target_image_pred = model(transformed_image.to(device))

    target_image_pred_probs = torch.softmax(target_image_pred, dim=1)
    target_image_pred_label = torch.argmax(target_image_pred_probs, dim=1)

    # Plot image with predicted label and probability
    plt.figure()
    plt.imshow(img)
    plt.title(
        f"Prediction:\t{class_names[target_image_pred_label]}\nProbability:\t{target_image_pred_probs.max():.3f}"
    )
    plt.axis(False)


In [ ]:
%%writefile {utils_script}
"""
Sample script for testing purposes
"""
# Imports
import os
import random
from datetime import datetime

import torch
from torch.utils.tensorboard import SummaryWriter


# Function to create a Tensorboard metric tracker
def create_writer(
    experiment_name: str, model_name: str, extra: str = None
) -> torch.utils.tensorboard.writer.SummaryWriter:
    """Creates a torch.utils.tensorboard.writer.SummaryWriter() instance saving to a specific log_dir.

    log_dir is a combination of runs/timestamp/experiment_name/model_name/extra.

    Where timestamp is the current date in YYYY-MM-DD format.

    Args:
        experiment_name (str): Name of experiment.
        model_name (str): Name of model.
        extra (str, optional): Anything extra to add to the directory. Defaults to None.

    Returns:
        torch.utils.tensorboard.writer.SummaryWriter(): Instance of a writer saving to log_dir.

    Example usage:
        # Create a writer saving to "runs/2022-06-04/data_10_percent/effnetb2/5_epochs/"
        writer = create_writer(experiment_name="data_10_percent",
                               model_name="effnetb2",
                               extra="5_epochs")
        # The above is the same as:
        writer = SummaryWriter(log_dir="runs/2022-06-04/data_10_percent/effnetb2/5_epochs/")
    """

    # Get timestamp of current date (all experiments on certain day live in same folder)
    timestamp = datetime.now().strftime(
        "%Y-%m-%d"
    )  # returns current date in YYYY-MM-DD format
    log_dir = os.path.join("runs", timestamp)

    if extra:
        # Create log directory path
        log_dir = os.path.join("runs", timestamp, experiment_name, model_name, extra)
    else:
        log_dir = os.path.join("runs", timestamp, experiment_name, model_name)

    print(f"[INFO] Created SummaryWriter, saving to: {log_dir}...")

    return SummaryWriter(log_dir=log_dir)


# Reset torch seeds for reproducibility
def reset_seed(seed: int = 42):
    """Sets random seeds for torch operations.

    Args:
        seed (int, optional): Random seed to set. Defaults to 42.
    """
    # Set seed for Python
    random.seed(seed)
    # Set CPU Torch Seed
    torch.manual_seed(seed)
    # Set CUDA Torch Seed
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    return

In [ ]:
%%writefile {app_script}
### 1. Imports and class names setup ###
import gradio as gr
import os
import torch

# from model import create_effnetb2_model
from timeit import default_timer as timer
from typing import Tuple, Dict

# Setup class names
class_names = ["pizza", "steak", "sushi"]

### 2. Model and transforms preparation ###

# Create EffNetB2 model
effnetb2, effnetb2_transforms = create_effnetb2_model(
    num_classes=3,  # len(class_names) would also work
)

# Load saved weights
effnetb2.load_state_dict(
    torch.load(
        f="09_pretrained_effnetb2_feature_extractor_pizza_steak_sushi_20_percent.pth",
        map_location=torch.device("cpu"),  # load to CPU
    )
)

### 3. Predict function ###


# Create predict function
def predict(img) -> Tuple[Dict, float]:
    """Transforms and performs a prediction on img and returns prediction and time taken."""
    # Start the timer
    start_time = timer()

    # Transform the target image and add a batch dimension
    img = effnetb2_transforms(img).unsqueeze(0)

    # Put model into evaluation mode and turn on inference mode
    effnetb2.eval()
    with torch.inference_mode():
        # Pass the transformed image through the model and turn the prediction logits into prediction probabilities
        pred_probs = torch.softmax(effnetb2(img), dim=1)

    # Create a prediction label and prediction probability dictionary for each prediction class (this is the required format for Gradio's output parameter)
    pred_labels_and_probs = {
        class_names[i]: float(pred_probs[0][i]) for i in range(len(class_names))
    }

    # Calculate the prediction time
    pred_time = round(timer() - start_time, 5)

    # Return the prediction dictionary and prediction time
    return pred_labels_and_probs, pred_time


### 4. Gradio app ###

# Create title, description and article strings
title = "FoodVision Mini 🍕🥩🍣"
description = "An EfficientNetB2 feature extractor computer vision model to classify images of food as pizza, steak or sushi."
article = "Created at [09. PyTorch Model Deployment](https://www.learnpytorch.io/09_pytorch_model_deployment/)."

# Create examples list from "examples/" directory
example_list = [["examples/" + example] for example in os.listdir("examples")]

# Create the Gradio demo
demo = gr.Interface(
    fn=predict,  # mapping function from input to output
    inputs=gr.Image(type="pil"),  # what are the inputs?
    outputs=[
        gr.Label(num_top_classes=3, label="Predictions"),  # what are the outputs?
        gr.Number(label="Prediction time (s)"),
    ],  # our fn has two outputs, therefore we have two outputs
    # Create examples list from "examples/" directory
    examples=example_list,
    title=title,
    description=description,
    article=article,
)

# Launch the demo!
demo.launch()

In [ ]:
%%writefile {requirements_file}
torch
torchvision
gradio

In [ ]:
%%writefile {requirements_script}
# Use to import to Colab Kernel Runtime
import ast, sys, subprocess
from pathlib import Path

# SCRIPTS_DIR = Path("TranslationModel")  # adjust if needed
SCRIPTS_DIR = Path.cwd()


def install_external_imports():
    # collect internal module names: script files AND directories
    internal = {p.stem for p in SCRIPTS_DIR.glob("*.py")}
    internal |= {p.name for p in SCRIPTS_DIR.iterdir() if p.is_dir()}

    for script in SCRIPTS_DIR.glob("*.py"):
        tree = ast.parse(script.read_text())
        pkgs = set()

        for node in ast.walk(tree):
            if isinstance(node, ast.Import):
                for n in node.names:
                    root = n.name.split(".")[0]
                    if root not in internal:
                        pkgs.add(root)
            elif isinstance(node, ast.ImportFrom) and node.module:
                root = node.module.split(".")[0]
                if root not in internal:
                    pkgs.add(root)

        for pkg in pkgs:
            try:
                __import__(pkg)
            except ImportError:
                subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])


if __name__ == "__main__":
    print(f"[INFO] Installing external imports from scripts in {SCRIPTS_DIR}...")
    install_external_imports()
    print("Finished installing external imports.")
    pass

In [ ]:
"""
from torchvision.datasets import Caltech101, Food101
from torchvision import transforms
from torch.utils.data import DataLoader

# optional transforms
transform = transforms.Compose(
    [
        transforms.Lambda(lambda img: img.convert("RGB")),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ]
)

# download + load training split
dataset = Food101(root="data", split="test", download=True, transform=transform)

# wrap in a DataLoader
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# example: iterate through a batch
for images, labels in loader:
    print(images.shape, labels.shape)
    break

print("Total samples:", len(dataset))
print("Number of classes:", len(dataset.classes))
print("Example classes:", dataset.classes[:10])
# print("Number of classes:", len(dataset.categories))
# print("Classes:", dataset.categories[:10], "...")
"""

In [ ]:
# %run {train_script}